In [ ]:
from collections import Counter
import json
from pathlib import Path
import shutil

LAST_NAME = "Antonio"
STUDENT_ID = "TUPM-26-1833"
SEED_DIGIT = 3
ID_SUM = 23

WORKSPACE = Path.cwd()
DATA_DIR = (
    WORKSPACE.parent / "data"
    if WORKSPACE.name == "Lab6_File_Handling_Additional_Exercises"
    else WORKSPACE / "data"
)
OUTPUT_DIR = WORKSPACE / "output"
BACKUP_DIR = WORKSPACE / "backup"

for d in [OUTPUT_DIR, BACKUP_DIR]:
  d.mkdir(parents=True, exist_ok=True)

dataset_path = DATA_DIR / "weblog.csv"
records_to_process = 1000 + ID_SUM  # 1023 records
target_status_class = (SEED_DIGIT % 4) + 2  # Target status class (5xx)

processed_records = 0
target_class_records = 0
status_counts = Counter()
url_counter = Counter()

filtered_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_filtered_logs.csv"

with open(dataset_path, "r", encoding="utf-8") as infile, open(
    filtered_csv_path, "w", encoding="utf-8"
) as outfile:

  # Read and write header
  header = infile.readline()
  outfile.write(header)

  for _ in range(records_to_process):
    line = infile.readline()
    if not line:
      break

    processed_records += 1
    fields = line.strip().split(",")

    if len(fields) >= 4:
      ip, time_str, url, status = fields[0], fields[1], fields[2], fields[3]

      if status.isdigit():
        status_code = int(status)
        status_counts[status_code] += 1

        if status_code // 100 == target_status_class:
          target_class_records += 1
          url_counter[url] += 1
          outfile.write(line)

top_5_urls = url_counter.most_common(5)
summary_data = {
    "student_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "records_processed": processed_records,
    "target_status_class": f"{target_status_class}xx",
    "records_in_target_class": target_class_records,
    "status_counts": dict(status_counts),
    "top_5_urls": top_5_urls,
}

json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_log_summary.json"
with open(json_path, "w", encoding="utf-8") as json_file:
  json.dump(summary_data, json_file, indent=4)

backup_path = BACKUP_DIR / json_path.name
shutil.copy(json_path, backup_path)
backup_exists = backup_path.exists()

print("--- Web Server Access Log Analysis Results ---")
print("Records processed:", processed_records)
print("Target status class:", f"{target_status_class}xx")
print("Records in target class:", target_class_records)
print("Count per status code:", dict(status_counts))
print("Five most requested URLs:", top_5_urls)
print("Filtered CSV file:", filtered_csv_path.name)
print("JSON summary file:", json_path.name)
print("Backup verification:", backup_exists)

--- Web Server Access Log Analysis Results ---
Records processed: 1023
Target status class: 5xx
Records in target class: 0
Count per status code: {200: 845, 302: 144, 304: 21, 206: 8, 404: 5}
Five most requested URLs: []
Filtered CSV file: Antonio_TUPM-26-1833_filtered_logs.csv
JSON summary file: Antonio_TUPM-26-1833_log_summary.json
Backup verification: True


In [ ]:
from collections import defaultdict
import csv
import json
from pathlib import Path
import shutil

LAST_NAME = "Antonio"
STUDENT_ID = "TUPM-26-1833"
SEED_DIGIT = 3

WORKSPACE = Path.cwd()
DATA_DIR = WORKSPACE.parent / "data"
OUTPUT_DIR = WORKSPACE / "output"
BACKUP_DIR = WORKSPACE / "backup"

for d in [OUTPUT_DIR, BACKUP_DIR]:
  d.mkdir(parents=True, exist_ok=True)

dataset_path = DATA_DIR / "user_behavior_dataset.csv"

target_os = "iOS"  # Since SEED_DIGIT (3) is odd
target_behavior_class = (SEED_DIGIT % 5) + 1  # Results in Class 4

total_records = 0
os_matches = 0
behavior_matches = 0

battery_sum = 0.0
data_usage_sum = 0.0
exported_rows_count = 0

filtered_csv_path = (
    OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_filtered_device_usage.csv"
)
exported_fields = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)",
]

with open(dataset_path, "r", encoding="utf-8") as infile, open(
    filtered_csv_path, "w", newline="", encoding="utf-8"
) as outfile:

  reader = csv.DictReader(infile)
  writer = csv.DictWriter(outfile, fieldnames=exported_fields)
  writer.writeheader()

  for row in reader:
    total_records += 1
    os_val = row.get("Operating System", "").strip()

    if os_val.lower() == target_os.lower():
      os_matches += 1

      # Check behavior class (handling integer/string parsing)
      try:
        b_class = int(float(row.get("User Behavior Class", 0)))
      except ValueError:
        b_class = -1

      if b_class == target_behavior_class:
        behavior_matches += 1

        # Accumulate metrics for averages
        try:
          battery_sum += float(
              row.get("Battery Drain (mAh/day)", 0)
          )  # adjust key name if exact CSV header differs
          data_usage_sum += float(row.get("Data Usage (MB/day)", 0))
        except ValueError:
          pass

        # Write matching fields
        filtered_row = {field: row.get(field, "") for field in exported_fields}
        writer.writerow(filtered_row)
        exported_rows_count += 1

# Calculate averages
avg_battery = (
    (battery_sum / behavior_matches) if behavior_matches > 0 else 0.0
)
avg_data_usage = (
    (data_usage_sum / behavior_matches) if behavior_matches > 0 else 0.0
)

summary_data = {
    "student_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "target_operating_system": target_os,
    "target_behavior_class": target_behavior_class,
    "total_records_in_dataset": total_records,
    "records_matching_os": os_matches,
    "records_in_target_behavior_class": behavior_matches,
    "average_battery_drain": round(avg_battery, 2),
    "average_data_usage": round(avg_data_usage, 2),
    "exported_row_count": exported_rows_count,
}

json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_device_summary.json"
with open(json_path, "w", encoding="utf-8") as json_file:
  json.dump(summary_data, json_file, indent=4)

# Backup JSON file and check CSV file size
backup_csv_path = BACKUP_DIR / filtered_csv_path.name
shutil.copy(filtered_csv_path, backup_csv_path)
csv_file_size = (
    filtered_csv_path.stat().st_size
    if filtered_csv_path.exists()
    else 0
)

# ---  PRINT RESULTS ---
print("--- Mobile Device Usage Profiling Results ---")
print("Total records in dataset:", total_records)
print(f"Records matching OS ({target_os}):", os_matches)
print(f"Records in target behavior class ({target_behavior_class}):", behavior_matches)
print("Average battery drain (mAh/day):", round(avg_battery, 2))
print("Average data usage (MB/day):", round(avg_data_usage, 2))
print("Count verification (Exported == Behavior matches):", exported_rows_count == behavior_matches)
print("Filtered CSV file:", filtered_csv_path.name)
print("JSON summary file:", json_path.name)
print("Backup file size (bytes):", csv_file_size)

--- Mobile Device Usage Profiling Results ---
Total records in dataset: 700
Records matching OS (iOS): 146
Records in target behavior class (4): 29
Average battery drain (mAh/day): 2123.21
Average data usage (MB/day): 1221.45
Count verification (Exported == Behavior matches): True
Filtered CSV file: Antonio_TUPM-26-1833_filtered_device_usage.csv
JSON summary file: Antonio_TUPM-26-1833_device_summary.json
Backup file size (bytes): 921


In [7]:
from collections import Counter, defaultdict
import csv
import json
from pathlib import Path
import shutil

LAST_NAME = "Antonio"
STUDENT_ID = "TUPM-26-1833"
SEED_DIGIT = 3

WORKSPACE = Path.cwd()
DATA_DIR = WORKSPACE.parent / "data"
OUTPUT_DIR = WORKSPACE / "output"
BACKUP_DIR = WORKSPACE / "backup"

for d in [OUTPUT_DIR, BACKUP_DIR]:
  d.mkdir(parents=True, exist_ok=True)

dataset_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
target_year = 2015 + SEED_DIGIT  # 2018

total_records = 0
target_year_records = 0
attack_counts = Counter()
total_loss = 0.0

incidents_list = []

with open(dataset_path, "r", encoding="utf-8") as infile:
  reader = csv.DictReader(infile)

  for row in reader:
    total_records += 1

    # Attempt to extract year from Date or Year column
    # (Adjust column name if the CSV uses 'Year' or 'Date')
    year_val = row.get("Year", "").strip()
    if not year_val and "Date" in row:
      # Extract year if stored as a full date string like YYYY-MM-DD
      year_val = row.get("Date", "").strip()[:4]

    try:
      row_year = int(year_val)
    except ValueError:
      row_year = -1

    if row_year == target_year:
      target_year_records += 1

      # Attack type count
      attack_type = row.get("Attack Type", "Unknown").strip()
      attack_counts[attack_type] += 1

      # Financial loss
      try:
        loss = float(row.get("Financial Loss (in Million $)", 0))
      except ValueError:
        loss = 0.0

      total_loss += loss

      # Store incident for sorting/top 5
      incident_record = dict(row)
      incident_record["_parsed_loss"] = loss
      incidents_list.append(incident_record)

# Calculate average financial loss
avg_loss = (total_loss / target_year_records) if target_year_records > 0 else 0.0

sorted_incidents = sorted(
    incidents_list, key=lambda x: x["_parsed_loss"], reverse=True
)
top_five = sorted_incidents[:5]

# Filtered CSV Export
filtered_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_filtered_incidents.csv"
if sorted_incidents:
  fieldnames = [
      col for col in sorted_incidents[0].keys() if col != "_parsed_loss"
  ]
  with open(filtered_csv_path, "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()
    for inc in sorted_incidents:
      clean_row = {k: v for k, v in inc.items() if k != "_parsed_loss"}
      writer.writerow(clean_row)

# JSON Summary Export
summary_data = {
    "student_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "target_year": target_year,
    "total_records_in_dataset": total_records,
    "incidents_in_target_year": target_year_records,
    "count_per_attack_type": dict(attack_counts),
    "total_financial_loss_million_usd": round(total_loss, 2),
    "average_financial_loss_million_usd": round(avg_loss, 2),
    "top_five_costliest_incidents": [
        {k: v for k, v in inc.items() if k != "_parsed_loss"}
        for inc in top_five
    ],
}

json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incident_summary.json"
with open(json_path, "w", encoding="utf-8") as json_file:
  json.dump(summary_data, json_file, indent=4)

# Backup JSON file & verify
backup_json_path = BACKUP_DIR / json_path.name
shutil.copy(json_path, backup_json_path)
backup_verified = backup_json_path.exists()

# Text Report Generation
report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cybersecurity_report.txt"
with open(report_path, "w", encoding="utf-8") as rep_file:
  rep_file.write(
      f"--- Cybersecurity Incident Report ({target_year}) ---\n"
  )
  rep_file.write(f"Student: {LAST_NAME} ({STUDENT_ID})\n")
  rep_file.write(f"Total Records: {total_records}\n")
  rep_file.write(f"Target Year Incidents: {target_year_records}\n")
  rep_file.write(f"Total Financial Loss: ${round(total_loss, 2)} Million\n")
  rep_file.write(f"Average Financial Loss: ${round(avg_loss, 2)} Million\n\n")
  rep_file.write("Attack Type Breakdown:\n")
  for atk, cnt in attack_counts.items():
    rep_file.write(f" - {atk}: {cnt}\n")

# ---  PRINT RESULTS ---
print("--- Cybersecurity Incident Report Results ---")
print("Target year:", target_year)
print("Total records in dataset:", total_records)
print("Incidents in target year:", target_year_records)
print("Count per attack type:", dict(attack_counts))
print("Total financial loss (USD millions):", round(total_loss, 2))
print("Average financial loss (USD millions):", round(avg_loss, 2))
print("Five costliest incidents found:", len(top_five))
print("Filtered CSV file:", filtered_csv_path.name)
print("JSON summary file:", json_path.name)
print("Backup verification (JSON copied):", backup_verified)
print("Report file:", report_path.name)

--- Cybersecurity Incident Report Results ---
Target year: 2018
Total records in dataset: 3000
Incidents in target year: 310
Count per attack type: {'Man-in-the-Middle': 36, 'SQL Injection': 57, 'Phishing': 52, 'DDoS': 60, 'Malware': 56, 'Ransomware': 49}
Total financial loss (USD millions): 14720.48
Average financial loss (USD millions): 47.49
Five costliest incidents found: 5
Filtered CSV file: Antonio_TUPM-26-1833_filtered_incidents.csv
JSON summary file: Antonio_TUPM-26-1833_incident_summary.json
Backup verification (JSON copied): True
Report file: Antonio_TUPM-26-1833_cybersecurity_report.txt
